## Question 2

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [2]:
## Import data
df = pd.read_csv('spambase/spambase.data', header=None)

columns = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free',
    'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650',
    'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857',
    'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project',
    'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!',
    'char_freq_$', 'char_freq_#',
    'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total', 'spam'
]

df.columns = columns

In [4]:
## Split data for training

X = df.drop(columns="spam")
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1
)

In [25]:
## Define GD for logistic regression

class LogisticRegressionGD:
    
    def __init__(self, alpha=0.01, iterations=100):
        self.alpha = alpha
        self.iterations = iterations
        self.theta = None
        self.losses = {}  
        
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def _cross_entropy_loss(self, y, y_hat):
        eps = 1e-15
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
        
    def fit(self, X, y, report_at={10, 50, 100}):
        n = X.shape[0]
        
        ones = np.ones((n, 1))
        X_b = np.hstack((ones, X))
        self.theta = np.zeros(X_b.shape[1])
        
        for i in range(1, self.iterations + 1):
            y_hat = self._sigmoid(X_b @ self.theta)
            errors = y_hat - y
            
            # Gradient
            gradient = (1/n) * (errors.T @ X_b)
            self.theta -= self.alpha * gradient
            
            if i in report_at:
                self.losses[i] = self._cross_entropy_loss(y, self._sigmoid(X_b @ self.theta))
            
    def predict_proba(self, X):
        n = X.shape[0]
        ones = np.ones((n, 1))
        X_b = np.hstack((ones, X))
        return self._sigmoid(X_b @ self.theta)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


In [26]:
## Define learning rates and train models

learning_rates = [0.01, 0.1, 0.5]
report_iters   = {10, 50, 100}
models_gd      = {}

for lr in learning_rates:
    m = LogisticRegressionGD(alpha=lr, iterations=100)
    m.fit(X_train, y_train, report_at=report_iters)
    models_gd[lr] = m

In [27]:
## Print Cross-Entropy loss for each value of iteration and learning rate
print("Cross-Entropy Loss")
for lr in learning_rates:
    print(f"\n  lr={lr}")
    for it in [10, 50, 100]:
        print(f"    iter {it}: {models_gd[lr].losses[it]:.6f}")


Cross-Entropy Loss

  lr=0.01
    iter 10: 12.364837
    iter 50: 12.354213
    iter 100: 13.954922

  lr=0.1
    iter 10: 19.663874
    iter 50: 12.557064
    iter 100: 12.516212

  lr=0.5
    iter 10: 20.838332
    iter 50: 13.213452
    iter 100: 13.625297


In [28]:
## Metrics after 100 iterations

print("\nMetrics at 100 Iterations")
for split_name, X_s, y_arr in [('Train', X_train, y_train),
                                 ('Test',  X_test,  y_test)]:
    print(f"\n  {split_name} Set")
    for lr in learning_rates:
        y_pred = models_gd[lr].predict(X_s)
        print(f"    lr={lr}  Acc={accuracy_score(y_arr, y_pred):.4f}  "
              f"Prec={precision_score(y_arr, y_pred):.4f}  "
              f"Rec={recall_score(y_arr, y_pred):.4f}  "
              f"F1={f1_score(y_arr, y_pred):.4f}")


Metrics at 100 Iterations

  Train Set
    lr=0.01  Acc=0.4258  Prec=0.4075  Rec=0.9985  F1=0.5788
    lr=0.1  Acc=0.5568  Prec=0.4604  Rec=0.7087  F1=0.5582
    lr=0.5  Acc=0.6055  Prec=0.7500  Rec=0.0022  F1=0.0044

  Test Set
    lr=0.01  Acc=0.4188  Prec=0.4021  Rec=1.0000  F1=0.5736
    lr=0.1  Acc=0.5769  Prec=0.4734  Rec=0.7311  F1=0.5747
    lr=0.5  Acc=0.6099  Prec=1.0000  Rec=0.0022  F1=0.0044


## Comparison

Comparing the different learning rates to each other, looking at the test set we see that 0.5 had the best accuracy and precision, however had terrible recall and F1 scores, showing that it is predicting almost nothing as spam, and the few times it does it happens to be correct, showing that this learning rate is too high. 0.01 has the opposite problem, with a recall of 1 and a lower precision showing that it is over predicting spam. 0.1 is a pretty good middle ground with okay numbers across the board. Comparing all of these numbers to the sklearn model however, this implementation performs pretty poorly. This could be due to that scaling that automatically happens in the sklearn model but does not happen in this implementation. This difference highlights the importance of scaling when implementing a Logistic Regression model and what the effects of not scaling have on the model.